# NanoGPT, continued
everything below picks up right after `BigramLanguageModel7`. the cells only need `text` (the tinyshakespeare string) from the top of the notebook, so they also run on their own.

**note:** this redefines `Block`, `GPT`-style classes with a config-based signature, so run the old `BigramLanguageModelN` cells *before* these if you still want them.

In [ ]:
# first the imports, and the device, this is new: until now everything silently lived on the cpu
import os, re, math, time, urllib.request
import numpy as np
from glob import glob
from itertools import islice
from urllib.parse import urlparse
from collections import Counter
from dataclasses import dataclass, asdict, replace
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# `text` is already loaded from the cells above, if you are running this cell alone we load it again
try:
    text
except NameError:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")
    with open("input.txt", 'r', encoding='utf-8') as f:
        text = f.read()
print(len(text))

In [ ]:
# before we build anything new, three things in the code above that were quietly wrong, we fix all of them below:
#
# 1. in train() we wrote `batch_size = 32`, but that creates a LOCAL variable, get_batch reads the global one, which is 4
#    so every model so far was trained on batches of 4 sequences, not 32. the config object removes this whole class of bug, there is only one batch_size
# 2. the models call torch.arange(T) with no device, as soon as the model is on a gpu this crashes, now every tensor is created on the right device
# 3. the attention scores were scaled by C**-0.5 where C is n_embd (32), but the scale should depend on head_size (8), the size of the vectors in the dot product
#    (the standalone demo even had head_size ** -5), the point of the scaling is to keep the variance of q @ k at 1 so softmax doesn't collapse to one-hot

## 1. a configuration system

In [ ]:
# first we want to get rid of the loose globals (n_embd, n_head, chunk_size...) that the classes were silently reading
# a dataclass gives us defaults, typos get caught immediately (cfg.n_embdd raises an error, config['n_embdd'] would not) and validation in one place
# I kept your name chunk_size for the context length

# before, every hyperparameter was a loose global (n_embd, n_head, chunk_size...) that the classes silently read from
# now they all live in one object that we pass around explicitly, so two models with different sizes can live side by side


@dataclass
class Config:
    # data
    data_path: str = "data/input.txt"  # a text file, a folder of .txt files, or an http(s) url (tinyshakespeare is downloaded if the default path is missing)

    # tokenizer
    tokenizer: str = "char"        # char | word | bpe
    bpe_backend: str = "python"    # bpe only: python (our own implementation) | hf (huggingface `tokenizers`, written in rust, much faster on big data)
    bpe_merges: int = 256          # bpe only: vocab size will be 256 bytes + this many merges
    word_min_freq: int = 3         # word only: words seen fewer times than this become <unk>

    # model
    chunk_size: int = 128          # the context length (called block_size in karpathy's code)
    n_embd: int = 128
    n_head: int = 4
    n_layer: int = 4
    dropout: float = 0.2
    activation: str = "gelu"       # relu | gelu
    pos_encoding: str = "learned"  # learned | rope
    pre_ln: bool = True            # True: x + sa(ln(x))  (what we had)   False: ln(x + sa(x))  (original transformer paper)
    weight_tying: bool = True      # share the token embedding matrix with the output layer
    flash: bool = True             # use pytorch's fused attention kernel when we don't need the attention weights

    # training
    batch_size: int = 64
    learning_rate: float = 3e-4
    min_lr_ratio: float = 0.1      # the decayed lr ends at learning_rate * min_lr_ratio
    max_iters: int = 5000
    lr_schedule: str = "warmup_cosine"  # constant | cosine | warmup_cosine
    warmup_iters: int = 200
    optimizer: str = "adamw"       # adam | adamw
    weight_decay: float = 0.1
    grad_clip: float = 1.0         # 0 disables clipping
    mixed_precision: bool = False  # gpu speedup 1: run the forward pass in 16 bit (bf16 on ampere+ gpus, fp16 + loss scaling on older ones like the kaggle T4)
    compile_model: bool = False    # gpu speedup 2: torch.compile fuses the model into faster kernels, costs a minute of compile time at the start (cuda only)

    # checkpoints
    ckpt_every_epochs: float = 0.0  # save a resumable checkpoint every N epochs (0 = off), can be fractional (0.5), "epoch" = as many tokens as the train set holds

    # evaluation
    eval_interval: int = 500
    eval_iters: int = 100
    patience: int = 3              # how many evals in a row val loss may stall (while train loss keeps dropping) before we flag overfitting
    early_stop: bool = False

    # misc
    seed: int = 1337
    use_wandb: bool = False

    def __post_init__(self):
        assert self.n_embd % self.n_head == 0, "n_embd must be divisible by n_head"
        assert self.tokenizer in ("char", "word", "bpe")
        assert self.bpe_backend in ("python", "hf")
        assert self.activation in ("relu", "gelu")
        assert self.pos_encoding in ("learned", "rope")
        assert self.lr_schedule in ("constant", "cosine", "warmup_cosine")
        assert self.optimizer in ("adam", "adamw")
        if self.pos_encoding == "rope":
            assert (self.n_embd // self.n_head) % 2 == 0, "rope rotates pairs of channels, so head_size must be even"

    def to_dict(self):
        return asdict(self)

    def but(self, **changes):
        # a copy of this config with a few fields changed, this is what makes experiments one-liners
        return replace(self, **changes)


# the model sizes we talked about in the ideas list
PRESETS = {
    "small":  dict(n_embd=64,  n_head=4, n_layer=2, chunk_size=64,  batch_size=32, dropout=0.1),
    "medium": dict(n_embd=128, n_head=4, n_layer=4, chunk_size=128, batch_size=64, dropout=0.2),
    "large":  dict(n_embd=256, n_head=8, n_layer=6, chunk_size=256, batch_size=64, dropout=0.2),
}


def load_config(path):
    import yaml  # imported here so the rest of the project works without pyyaml
    with open(path, "r") as f:
        return Config(**yaml.safe_load(f))

cfg = Config(**PRESETS['medium'])
print(cfg)
print(cfg.but(n_layer=6).n_layer, cfg.n_layer)  # but() returns a modified copy, this is how experiments change exactly one thing

## 2. tokenizers

In [ ]:
# until now the vocabulary was the 65 characters of the dataset, to compare tokenizers we need 3 of them behind the same interface: encode(str) -> list, decode(list) -> str
# the word and bpe tokenizers share one pre-tokenizer that keeps the space attached to the word, so decode is a plain join and nothing is lost

# one shared pre-tokenizer for the word and bpe tokenizers: a word together with its leading space, a single punctuation mark, or a run of whitespace
# keeping the space glued to the word means decode is just ''.join(...) and nothing is ever lost
PRETOKEN = re.compile(r" ?\w+| ?[^\w\s]|\s+")


class CharTokenizer:
    """ every character is a token, this is exactly what we built at the start of the notebook """

    def __init__(self, text):
        chars = sorted(list(set(text)))
        self.word2num = {ch: i for i, ch in enumerate(chars)}
        self.num2word = {i: ch for i, ch in enumerate(chars)}
        self.vocab_size = len(chars)
        # lookup table indexed by unicode code point, lets us encode hundreds of MB with numpy instead of a python loop
        self.lut = np.full(max(ord(c) for c in chars) + 1, -1, dtype=np.int32)
        for c, i in self.word2num.items():
            self.lut[ord(c)] = i

    def encode(self, s):
        return [self.word2num[c] for c in s]

    def encode_np(self, s):
        cp = np.frombuffer(s.encode('utf-32-le'), dtype=np.uint32)  # the code point of every character, in one go
        if len(cp) and int(cp.max()) >= len(self.lut):
            raise KeyError("character not in vocabulary")
        ids = self.lut[cp]
        if (ids < 0).any():
            raise KeyError("character not in vocabulary")
        return ids

    def decode(self, ids):
        return ''.join(self.num2word[i] for i in ids)


class WordTokenizer:
    """ every (space + word) is a token, rare words collapse into <unk> so the vocabulary doesn't explode """

    def __init__(self, text, min_freq=3):
        counts = Counter(PRETOKEN.findall(text))
        words = ['<unk>'] + sorted(w for w, c in counts.items() if c >= min_freq)  # <unk> sits at index 0
        self.word2num = {w: i for i, w in enumerate(words)}
        self.num2word = {i: w for i, w in enumerate(words)}
        self.vocab_size = len(words)

    def encode(self, s):
        return [self.word2num.get(w, 0) for w in PRETOKEN.findall(s)]  # anything we haven't seen before maps to <unk>

    def decode(self, ids):
        return ''.join(self.num2word[i] for i in ids)


class BPETokenizer:
    """ byte pair encoding: start from raw bytes and repeatedly merge the most frequent neighbouring pair into a new token """

    def __init__(self, text, num_merges=256):
        word_counts = Counter(PRETOKEN.findall(text))
        # merges never cross word boundaries, so we only need to train on the unique words weighted by how often they appear
        # (this makes training seconds instead of minutes in pure python)
        words = {w: list(w.encode('utf-8')) for w in word_counts}
        self.merges = {}  # (id_a, id_b) -> new id, the order of insertion is the order of priority
        self.vocab = {i: bytes([i]) for i in range(256)}
        for n in range(num_merges):
            pairs = Counter()
            for w, ids in words.items():
                for p in zip(ids, ids[1:]):
                    pairs[p] += word_counts[w]
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            new_id = 256 + n
            self.merges[best] = new_id
            self.vocab[new_id] = self.vocab[best[0]] + self.vocab[best[1]]
            for w in words:
                words[w] = self._merge(words[w], best, new_id)
        self.vocab_size = 256 + len(self.merges)
        self.cache = {}

    @staticmethod
    def _merge(ids, pair, new_id):
        out, i = [], 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
                out.append(new_id)
                i += 2
            else:
                out.append(ids[i])
                i += 1
        return out

    def _encode_word(self, w):
        if w in self.cache:
            return self.cache[w]
        ids = list(w.encode('utf-8'))
        while len(ids) >= 2:
            # always apply the merge that was learned first (lowest id), that is what makes the encoding deterministic
            best = min(set(zip(ids, ids[1:])), key=lambda p: self.merges.get(p, float('inf')))
            if best not in self.merges:
                break
            ids = self._merge(ids, best, self.merges[best])
        self.cache[w] = ids
        return ids

    def encode(self, s):
        return [i for w in PRETOKEN.findall(s) for i in self._encode_word(w)]

    def decode(self, ids):
        return b''.join(self.vocab[i] for i in ids).decode('utf-8', errors='replace')


def _split_at_newlines(text, size=1_000_000):
    # ~1MB pieces, always cut right after a newline so no word is ever split in half
    i = 0
    while i < len(text):
        j = min(len(text), i + size)
        if j < len(text):
            k = text.find("\n", j)
            j = len(text) if k == -1 else k + 1
        yield text[i:j]
        i = j


class HFBPETokenizer:
    """ the same byte-level bpe idea, but trained and run by huggingface's rust `tokenizers` library (pip install tokenizers) """

    def __init__(self, text, num_merges=256):
        from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers  # imported here so the project works without it
        tok = Tokenizer(models.BPE())
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)  # bytes first, exactly like our own implementation, so nothing is ever unknown
        tok.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(vocab_size=256 + num_merges, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), special_tokens=[], show_progress=False)
        tok.train_from_iterator(_split_at_newlines(text), trainer)
        self.tok = tok
        self.vocab_size = tok.get_vocab_size()

    def encode_np(self, s):
        # encode_batch hands back every token id as a python int, ~30 bytes each, so on a few hundred MB of text we must convert to numpy (4 bytes each) in small groups
        # instead of holding all of it at once, otherwise the notebook runs out of ram
        pieces = _split_at_newlines(s)
        out = []
        while True:
            group = list(islice(pieces, 32))  # 32 pieces of ~1MB
            if not group:
                break
            out.extend(np.array(e.ids, dtype=np.int32) for e in self.tok.encode_batch(group))  # encode_batch runs the pieces in parallel on all cpu cores
        return np.concatenate(out) if out else np.zeros(0, dtype=np.int32)

    def encode(self, s):
        return self.encode_np(s).tolist()

    def decode(self, ids):
        return self.tok.decode(list(ids))

    # the rust object can't be pickled directly, so the checkpoint stores it as a json string and rebuilds it on load
    def __getstate__(self):
        return {'json': self.tok.to_str(), 'vocab_size': self.vocab_size}

    def __setstate__(self, state):
        from tokenizers import Tokenizer
        self.tok = Tokenizer.from_str(state['json'])
        self.vocab_size = state['vocab_size']


def get_tokenizer(cfg, text):
    if cfg.tokenizer == "char":
        return CharTokenizer(text)
    if cfg.tokenizer == "word":
        return WordTokenizer(text, cfg.word_min_freq)
    if cfg.bpe_backend == "hf":
        return HFBPETokenizer(text, cfg.bpe_merges)
    return BPETokenizer(text, cfg.bpe_merges)

In [ ]:
# the data preparation, two decisions here:
# - we split the TEXT 90/10 and only then tokenize, so all tokenizers are validated on exactly the same characters
# - the tokenizer (vocab or merges) is learned from the train part only, otherwise val would leak into the model

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DEFAULT_PATH = "data/input.txt"
_cache = {}  # building a tokenizer can take a while, so experiments that reuse one shouldn't pay for it twice


def _download(url, local):
    os.makedirs(os.path.dirname(local) or ".", exist_ok=True)
    if not os.path.exists(local):
        urllib.request.urlretrieve(url, local)
    return local


def load_text(path=None):
    # the training data can now be: a single text file, a folder (every .txt file inside, subfolders included, is joined), or an http(s) url
    path = path or DEFAULT_PATH
    if path.startswith(("http://", "https://")):
        path = _download(path, os.path.join("data", os.path.basename(urlparse(path).path) or "download.txt"))
    if os.path.isdir(path):
        files = sorted(glob(os.path.join(path, "**", "*.txt"), recursive=True))
        if not files:
            raise FileNotFoundError(f"no .txt files found inside {path}")
        parts = []
        for fp in files:
            with open(fp, 'r', encoding='utf-8', errors='replace') as f:  # real-world corpora always contain a few broken bytes, we replace them instead of crashing
                parts.append(f.read())
        print(f"loaded {len(files)} files from {path}")
        return "\n\n".join(parts)
    if not os.path.exists(path):
        if path == DEFAULT_PATH:
            _download(URL, path)  # the original tinyshakespeare, same as the !wget in the notebook
        else:
            raise FileNotFoundError(f"data path not found: {path}")
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        return f.read()


def encode_tensor(tokenizer, text):
    # int32 instead of int64 halves the memory of the dataset (matters at 100M+ tokens), get_batch converts the small batches back to long
    if hasattr(tokenizer, 'encode_np'):
        arr = tokenizer.encode_np(text)  # the char and huggingface tokenizers have a fast numpy path
    else:
        arr = np.array(tokenizer.encode(text), dtype=np.int32)
    return torch.from_numpy(np.ascontiguousarray(arr, dtype=np.int32))


def prepare(cfg, text):
    # we split the TEXT 90/10 first and only then tokenize, so every tokenizer is validated on exactly the same characters
    # (splitting the token stream would give each tokenizer a slightly different validation set, and then we couldn't compare them)
    key = (cfg.tokenizer, cfg.bpe_backend, cfg.bpe_merges, cfg.word_min_freq, len(text))
    if key not in _cache:
        split = int(0.9 * len(text))
        train_text, val_text = text[:split], text[split:]
        # the vocab/merges are learned from the train part only, no peeking at val
        # except for the char tokenizer: its "vocabulary" is just the alphabet, and on a big corpus a rare character that only shows up in val would crash the encoding
        tokenizer = get_tokenizer(cfg, text if cfg.tokenizer == 'char' else train_text)
        train_data = encode_tensor(tokenizer, train_text)
        val_data = encode_tensor(tokenizer, val_text)
        _cache[key] = (tokenizer, train_data, val_data, len(val_text))
    return _cache[key]  # tokenizer, train_data, val_data, number of characters in val


def get_batch(data, cfg, device):
    ix = torch.randint(low=0, high=len(data) - cfg.chunk_size - 1, size=(cfg.batch_size,))
    x = torch.stack([data[i: i + cfg.chunk_size] for i in ix])
    y = torch.stack([data[i + 1: i + cfg.chunk_size + 1] for i in ix])
    return x.to(device).long(), y.to(device).long()  # the tensors have to live on the same device as the model, and embeddings want int64

In [ ]:
# now let us see what each tokenizer does to the same data
# bpe has two backends: our own pure python one (backend='python') and huggingface's rust one (backend='hf', needs: pip install tokenizers)
# same algorithm, same vocab size, the difference is speed, which starts to matter once the dataset is not 1MB anymore
sentence = "To be, or not to be, that is the question."
for t, backend in [('char', 'python'), ('word', 'python'), ('bpe', 'python'), ('bpe', 'hf')]:
    t0 = time.time()
    try:
        tok, tr, va, nchars = prepare(Config(tokenizer=t, bpe_backend=backend), text)
    except ImportError:
        print(f"{t} ({backend}): pip install tokenizers to try this one"); continue
    ids = tok.encode(sentence)
    print(f"{t:5s} ({backend:6s}) vocab {tok.vocab_size:5d} | train tokens {len(tr):8d} | chars/token {nchars/len(va):.2f} | sentence -> {len(ids)} tokens | roundtrip ok: {tok.decode(ids) == sentence} | {time.time()-t0:.1f}s")
print([tok.decode([i]) for i in tok.encode(sentence)])  # the bpe pieces

## 3. the model, upgraded

In [ ]:
# attention first. three changes here:
# - all heads in one module (one big qkv matrix instead of 3 * n_head tiny ones), the maths is the same as Head + MultiHeadAttention but it is a lot faster
# - the attention weights can be kept (keep_attn) for the visualization, and a kv cache can be passed in
# - optional rotary position embeddings, and pytorch's fused attention kernel when we don't need the weights

def rope_cos_sin(positions, head_size, base=10000.0):
    # rotary embeddings: instead of ADDING a position vector to the input, we ROTATE the queries and keys by an angle that depends on the position
    # every pair of channels (i, i + head_size/2) gets its own rotation speed, positions: (T,) absolute positions
    freqs = 1.0 / (base ** (torch.arange(0, head_size, 2, device=positions.device).float() / head_size))  # (head_size/2,)
    angles = positions.float()[:, None] * freqs[None, :]  # (T, head_size/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x is (B, H, T, head_size), we rotate each (x1, x2) pair, the dot product q.k then only depends on the RELATIVE distance between the two positions
    half = x.size(-1) // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)  # cos/sin are float32, under mixed precision we must give back the dtype we got


class CausalSelfAttention(nn.Module):
    """ all the heads of self-attention in one module, same idea as Head + MultiHeadAttention but computed in parallel """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_size = cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)  # one big matrix instead of 3 * n_head tiny ones, the heads are just slices of it
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.dropout_p = cfg.dropout
        self.flash = cfg.flash
        self.register_buffer('tril', torch.tril(torch.ones(cfg.chunk_size, cfg.chunk_size)))
        self.last_attn = None  # only filled when we ask for it, this is what the visualization reads

    def forward(self, x, rope=None, past=None, keep_attn=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, H, T, head_size)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        if rope is not None:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)  # note that v is NOT rotated, only the things used to compute the affinities

        if past is not None:
            # kv cache: the keys and values of the tokens we already processed are glued in front of the new ones
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v)
        Tk = k.size(2)  # number of keys = cached tokens + new tokens

        if self.flash and not keep_attn and (past is None or T == 1):
            # fused kernel, never builds the (T, T) matrix in memory, with a single new token no mask is needed since it may look at everything before it
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout_p if self.training else 0.0, is_causal=(past is None))
        else:
            wei = (q @ k.transpose(-2, -1)) * self.head_size ** -0.5  # (B, H, T, Tk), scaled by head_size, the size of the vectors being dotted
            wei = wei.masked_fill(self.tril[Tk - T:Tk, :Tk] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            if keep_attn:
                self.last_attn = wei.detach()
            wei = self.attn_dropout(wei)
            out = wei @ v  # (B, H, T, head_size)

        out = out.transpose(1, 2).contiguous().view(B, T, C)  # glue the heads back together, this is the torch.cat we used to do
        return self.resid_dropout(self.proj(out)), present

In [ ]:
# the block, the feed forward gets a GELU option and the block gets a pre/post layernorm switch
# (your Block was already pre-ln: x + sa(ln(x)), post-ln is the original paper's version, so now we can measure the difference)

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, cfg):
        super().__init__()
        act = nn.GELU() if cfg.activation == "gelu" else nn.ReLU()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            act,
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, cfg):
        super().__init__()
        self.sa = CausalSelfAttention(cfg)
        self.ffwd = FeedForward(cfg)
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.pre_ln = cfg.pre_ln

    def forward(self, x, rope=None, past=None, keep_attn=False):
        if self.pre_ln:
            a, present = self.sa(self.ln1(x), rope, past, keep_attn)  # normalize BEFORE the sublayer, the residual stream itself stays untouched
            x = x + a
            x = x + self.ffwd(self.ln2(x))
        else:
            a, present = self.sa(x, rope, past, keep_attn)  # normalize AFTER adding, like in "attention is all you need"
            x = self.ln1(x + a)
            x = self.ln2(x + self.ffwd(x))
        return x, present  # present is the (keys, values) of this layer for the kv cache

In [ ]:
# the whole model, this replaces the BigramLanguageModelN family, it takes a config and a vocab size and nothing else
# new things: weight tying, the gpt-2 style init, rope vs learned positions, and a generate() that can use a kv cache

class GPT(nn.Module):
    """ the final model, this replaces the whole BigramLanguageModel family, it hasn't been a bigram model for a while now """

    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.cfg = cfg
        self.token_embedding_table = nn.Embedding(vocab_size, cfg.n_embd)
        self.position_embedding_table = nn.Embedding(cfg.chunk_size, cfg.n_embd) if cfg.pos_encoding == "learned" else None  # rope has no table at all
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])  # ModuleList instead of Sequential, since each block now also takes and returns the kv cache
        self.ln_f = nn.LayerNorm(cfg.n_embd)  # final layer norm
        self.lm_head = nn.Linear(cfg.n_embd, vocab_size, bias=False)
        if cfg.weight_tying:
            # the embedding maps token -> vector, the lm_head maps vector -> token, the same matrix can do both jobs
            self.lm_head.weight = self.token_embedding_table.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith('sa.proj.weight') or name.endswith('ffwd.net.2.weight'):
                # every block adds two things to the residual stream, so we shrink the layers that write to it, otherwise its variance grows with depth
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, past=None, use_cache=False, keep_attn=False):
        # idx and targets are both (B, T) tensor of integers
        B, T = idx.shape
        past_len = past[0][0].size(2) if past is not None else 0
        assert past_len + T <= self.cfg.chunk_size, "sequence is longer than the context the model was trained with"
        positions = torch.arange(past_len, past_len + T, device=idx.device)  # with a cache the new token isn't at position 0 anymore

        x = self.token_embedding_table(idx)  # (B,T,C)
        rope = None
        if self.cfg.pos_encoding == "learned":
            x = x + self.position_embedding_table(positions)  # (B,T,C) + (T,C)
        else:
            rope = rope_cos_sin(positions, self.cfg.n_embd // self.cfg.n_head)
        x = self.drop(x)

        presents = []
        for i, block in enumerate(self.blocks):
            x, present = block(x, rope, past[i] if past is not None else None, keep_attn)
            presents.append(present)
        x = self.ln_f(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))

        if use_cache:
            return logits, loss, presents
        return logits, loss  # same output as before, so train loops written for the old models still work

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, use_cache=True, temperature=1.0, top_k=None):
        # idx is (B, T) array of indices in the current context
        was_training = self.training
        self.eval()  # generation shouldn't use dropout
        chunk_size = self.cfg.chunk_size
        past = None
        for _ in range(max_new_tokens):
            if not use_cache:
                # the old way: crop to the last chunk_size tokens and recompute everything from scratch, every single step
                logits, _ = self(idx[:, -chunk_size:])
            else:
                if past is None:
                    logits, _, past = self(idx[:, -chunk_size:], use_cache=True)  # first step: process the whole prompt once and fill the cache
                elif past[0][0].size(2) >= chunk_size:
                    # the window is full, positions would go out of range, so we rebuild the cache from the last half of the context
                    # (rebuilding from half and not from scratch means we only pay for it once every chunk_size/2 tokens)
                    logits, _, past = self(idx[:, -(chunk_size // 2):], use_cache=True)
                else:
                    logits, _, past = self(idx[:, -1:], past=past, use_cache=True)  # only the newest token goes in, everything else is in the cache
            logits = logits[:, -1, :]  # becomes (B, C)
            if temperature == 0:
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # temperature -> 0 is the limit of "always take the most likely token" (greedy), dividing by 0 would give nan
            else:
                logits = logits / temperature  # <1 sharpens the distribution (safer, more repetitive), >1 flattens it (more surprising, more typos)
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))  # the k best logits per row, sorted
                    logits = logits.masked_fill(logits < v[:, [-1]], float('-inf'))  # everything below the k-th best can never be sampled
                probs = F.softmax(logits, dim=-1)  # (B, C)
                idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        self.train(was_training)
        return idx

    @torch.no_grad()
    def attention_maps(self, idx):
        # returns the attention weights of every layer and head for the first sequence in idx: (n_layer, n_head, T, T)
        was_training = self.training
        self.eval()
        self(idx[:, -self.cfg.chunk_size:], keep_attn=True)
        maps = torch.stack([b.sa.last_attn[0] for b in self.blocks])
        self.train(was_training)
        return maps.cpu()

In [ ]:
# generation helpers: generate_text, a check that the cache is exact, and a benchmark

def load_checkpoint(path, device='cpu'):
    # the checkpoint holds the weights, the config AND the tokenizer, so a model can always be rebuilt from one file
    # (weights_only=False because we pickle the tokenizer, only load checkpoints you made yourself)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = Config(**ckpt['config'])
    tokenizer = ckpt['tokenizer']
    model = GPT(cfg, tokenizer.vocab_size).to(device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    model.metrics = ckpt.get('metrics', {})  # val loss / bits per char / params saved at training time, the app shows them
    return model, tokenizer, cfg


def generate_text(model, tokenizer, prompt="\n", max_new_tokens=500, device='cpu', use_cache=True, temperature=1.0, top_k=None):
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)  # the default prompt is a newline, which is token 0 for the char model, just like torch.zeros before
    out = model.generate(idx, max_new_tokens, use_cache=use_cache, temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0].tolist())


@torch.no_grad()
def verify_cache(model, device='cpu', atol=1e-4):
    # same idea as our torch.allclose checks at the start: the cache must change the speed, never the answer
    # we feed a sequence token by token through the cache, and compare against one full forward pass
    model.eval()
    T = model.cfg.chunk_size
    idx = torch.randint(0, model.token_embedding_table.num_embeddings, (1, T), device=device)
    full, _ = model(idx)
    past, steps = None, []
    for t in range(T):
        logits, _, past = model(idx[:, t:t + 1], past=past, use_cache=True)
        steps.append(logits)
    return torch.allclose(full, torch.cat(steps, dim=1), atol=atol)


@torch.no_grad()
def benchmark_kv_cache(model, new_tokens=None, device='cpu'):
    # the cached version is exactly the same math, so the only thing that changes is how long it takes
    model.eval()
    new_tokens = new_tokens or model.cfg.chunk_size - 1  # stay inside one context window so the comparison is clean
    idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    results = {}
    for use_cache in (False, True):
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.time()
        model.generate(idx, new_tokens, use_cache=use_cache)
        if device == 'cuda':
            torch.cuda.synchronize()
        results[use_cache] = time.time() - t0
        print(f"use_cache={use_cache}: {new_tokens} tokens in {results[use_cache]:.2f}s ({new_tokens / results[use_cache]:.0f} tok/s)")
    print(f"speedup: {results[False] / results[True]:.2f}x")
    return results

In [ ]:
# sanity checks in the same spirit as our torch.allclose cells, new machinery must not change the answers
torch.manual_seed(1337)
small = Config(n_embd=64, n_head=4, n_layer=2, chunk_size=32, dropout=0.0)

for name, c in [('learned pos', small), ('rope', small.but(pos_encoding='rope')), ('post-ln, relu, untied', small.but(pre_ln=False, activation='relu', weight_tying=False))]:
    m = GPT(c, 65).to(device).eval()
    x = torch.randint(0, 65, (2, 32), device=device)
    a, _ = m(x)
    for b in m.blocks: b.sa.flash = False          # switch to our manual attention
    b_, _ = m(x)
    print(f"{name:24s} fused == manual: {torch.allclose(a, b_, atol=1e-4)} | kv cache == full forward: {verify_cache(m, device)}")

print('params tied vs untied:', sum(p.numel() for p in GPT(small, 65).parameters()), sum(p.numel() for p in GPT(small.but(weight_tying=False), 65).parameters()))

## 4. a proper evaluation system

In [ ]:
# so far we judged the model by the loss printed in the training loop and by reading generated text, this is where we do it properly
# - the loss of one batch is noise, estimate_loss averages many batches, with dropout off, for both train and val
# - perplexity = e^loss
# - bits per char is the one metric that lets us compare tokenizers, perplexity is per token and a bpe token is worth ~2 characters
# - the overfit detector looks at the gap between train and val

@torch.no_grad()  # no gradients needed here, this saves memory and time
def estimate_loss(model, train_data, val_data, cfg, device, amp_ctx=None):
    # the loss of a single batch is very noisy, so we average over eval_iters batches, for both splits
    # eval mode turns dropout off, so train and val are measured under the same conditions (the running train loss we print inside the loop has dropout on)
    amp_ctx = amp_ctx or (lambda: torch.autocast(device_type='cpu', enabled=False))  # a context factory, so evaluation uses the same precision as training
    out = {}
    model.eval()
    for split, data in [('train', train_data), ('val', val_data)]:
        losses = torch.zeros(cfg.eval_iters)
        for k in range(cfg.eval_iters):
            x, y = get_batch(data, cfg, device)
            with amp_ctx():
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def perplexity(loss):
    # ppl = e^(cross entropy), "the model is as unsure as if it had to pick uniformly between ppl tokens"
    return math.exp(loss)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())  # parameters() already skips duplicates, so tied weights are counted once


def bits_per_char(loss, n_tokens, n_chars):
    # perplexity is per TOKEN, so a char model and a bpe model can't be compared with it (a bpe token is worth ~4 chars)
    # total nats = loss * n_tokens, we divide by the characters those tokens cover, and by ln(2) to get bits
    return loss * n_tokens / n_chars / math.log(2)


class OverfitDetector:
    """ flags overfitting when val loss stops improving for `patience` evals in a row while the train loss is still going down """

    def __init__(self, patience=3, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.best_val = float('inf')
        self.prev_train = float('inf')
        self.bad_evals = 0

    def update(self, train_loss, val_loss):
        if val_loss < self.best_val - self.min_delta:
            self.best_val = val_loss
            self.bad_evals = 0
        elif train_loss < self.prev_train:  # val is stuck/worse but train still improves -> the gap is growing
            self.bad_evals += 1
        self.prev_train = train_loss
        return self.bad_evals >= self.patience

## 5. optimization

In [ ]:
# adamw vs adam, weight decay only on matrices, gradient clipping, and the lr schedule as a plain function of the step so we can plot it before spending any gpu time

def configure_optimizer(model, cfg):
    # weight decay should only shrink the big weight matrices (dim >= 2: linear layers, embeddings)
    # biases and layernorm gains (dim < 2) are left alone, decaying them only hurts
    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
    groups = [{'params': decay, 'weight_decay': cfg.weight_decay},
              {'params': no_decay, 'weight_decay': 0.0}]
    if cfg.optimizer == "adamw":
        return torch.optim.AdamW(groups, lr=cfg.learning_rate)  # decay is applied directly to the weights, separate from the gradient
    return torch.optim.Adam(groups, lr=cfg.learning_rate)  # here the same number becomes an L2 penalty added to the gradient, which adam then rescales


def get_lr(it, cfg):
    # we compute the lr by hand every step instead of using torch schedulers, a plain function of the step is easy to plot and to reason about
    if cfg.lr_schedule == "constant":
        return cfg.learning_rate
    min_lr = cfg.learning_rate * cfg.min_lr_ratio
    warmup = cfg.warmup_iters if cfg.lr_schedule == "warmup_cosine" else 0
    if it < warmup:
        return cfg.learning_rate * (it + 1) / warmup  # linear warmup, adam's statistics are garbage in the first steps so we start gently
    progress = min(1.0, (it - warmup) / max(1, cfg.max_iters - warmup))
    return min_lr + 0.5 * (1 + math.cos(math.pi * progress)) * (cfg.learning_rate - min_lr)  # cosine from learning_rate down to min_lr

c = Config(max_iters=5000, warmup_iters=200)
steps = range(c.max_iters + 1)
for sched in ['constant', 'cosine', 'warmup_cosine']:
    plt.plot(steps, [get_lr(i, c.but(lr_schedule=sched)) for i in steps], label=sched)
plt.xlabel('step'); plt.ylabel('learning rate'); plt.legend(); plt.show()

## 6. the training loop

In [ ]:
# the old train() printed the (noisy, dropout-on) loss of one batch every 1000 steps, the new one evaluates properly, measures throughput, keeps the best weights and watches for overfitting
# tokens/sec only counts the training steps, the evaluation time is excluded

def get_device():
    return 'cuda' if torch.cuda.is_available() else 'cpu'


def amp_settings(cfg, device):
    # which 16 bit format to use, and whether we need loss scaling
    device_type = 'cuda' if 'cuda' in device else 'cpu'
    if device_type == 'cuda':
        # bf16 has the same range as fp32 so it needs nothing special, but only ampere gpus (compute capability 8+) run it natively, the kaggle T4/P100 don't
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dtype = torch.bfloat16  # cpu autocast only really makes sense in bf16, and only pays off on newer cpus
    use_scaler = cfg.mixed_precision and dtype == torch.float16  # fp16 can't represent tiny gradients (they become 0), so we multiply the loss by a big number first and divide the gradients afterwards
    return device_type, dtype, use_scaler


def save_checkpoint(path, weights, cfg, tokenizer, metrics, optimizer=None, step=None, history=None):
    # the final checkpoint is inference-only (small), the periodic ones also carry the optimizer state and the step so training can be resumed from them
    ckpt = {'model': weights, 'config': cfg.to_dict(), 'tokenizer': tokenizer, 'metrics': metrics}
    if optimizer is not None:
        ckpt.update(optimizer=optimizer.state_dict(), step=step, history=history)
    torch.save(ckpt, path)


def train_model(cfg, tokenizer, train_data, val_data, val_chars, device, name="run", verbose=True, ckpt_path=None, resume_from=None):
    torch.manual_seed(cfg.seed)
    model = GPT(cfg, tokenizer.vocab_size).to(device)
    n_params = count_parameters(model)
    optimizer = configure_optimizer(model, cfg)
    detector = OverfitDetector(cfg.patience)
    history = {k: [] for k in ['step', 'train_loss', 'val_loss', 'val_ppl', 'val_bpc', 'lr', 'tokens_per_sec', 'elapsed']}
    history.update(name=name, n_params=n_params, overfit_step=None, best_val=float('inf'))

    start_it = 0
    if resume_from:
        ckpt = torch.load(resume_from, map_location=device, weights_only=False)
        assert 'optimizer' in ckpt, "this checkpoint has no optimizer state, only the periodic epoch checkpoints can be resumed"
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        history = ckpt['history']
        history['best_val'] = float('inf')  # the best weights are tracked from the resume point on
        start_it = ckpt['step'] + 1
        assert start_it <= cfg.max_iters, f"checkpoint is already at step {ckpt['step']}, raise max_iters to keep training"
    torch.manual_seed(cfg.seed + start_it)  # a resumed run must not replay the same batches it already saw

    # how many training steps are "one epoch", batches are random so this is the number of tokens in the train set divided by the tokens per step
    iters_per_epoch = len(train_data) / (cfg.batch_size * cfg.chunk_size)
    ckpt_every = max(1, round(cfg.ckpt_every_epochs * iters_per_epoch)) if (cfg.ckpt_every_epochs > 0 and ckpt_path) else 0

    # gpu speedups, both are opt-in through the config
    device_type, amp_dtype, use_scaler = amp_settings(cfg, device)
    amp_ctx = lambda: torch.autocast(device_type=device_type, dtype=amp_dtype, enabled=cfg.mixed_precision)
    scaler = torch.amp.GradScaler(device_type, enabled=use_scaler)  # when it is disabled, scale/unscale/step/update are plain pass-throughs, so there is one code path for everything
    fwd = model  # evaluation and generation always use the plain model, only the training steps use the compiled one
    if cfg.compile_model:
        if device_type != 'cuda':
            print("compile_model is only used on cuda, continuing without it")
        elif torch.cuda.get_device_capability()[0] < 7:
            # the compiler (triton) needs a volta/turing/newer gpu, and the error would only appear at the first training step, so we check up front (kaggle's P100 is too old)
            print(f"compile_model needs a gpu with compute capability 7+, this one has {torch.cuda.get_device_capability()[0]}.x, continuing without it")
        else:
            try:
                fwd = torch.compile(model)  # the compiled wrapper shares its parameters with `model`, so checkpoints stay clean
            except Exception as e:
                print(f"torch.compile failed ({e}), continuing without it")

    if verbose:
        prec = str(amp_dtype).replace('torch.', '') if cfg.mixed_precision else 'fp32'
        print(f"[{name}] {n_params/1e6:.3f}M parameters, vocab {tokenizer.vocab_size}, device {device}, precision {prec}, compile {fwd is not model}, "
              f"{iters_per_epoch:.0f} steps/epoch ({cfg.max_iters / iters_per_epoch:.1f} epochs total)" + (f", resuming at step {start_it}" if start_it else ""))

    if cfg.use_wandb:
        import wandb
        wandb.init(project="nano-gpt", name=name, config={**cfg.to_dict(), 'n_params': n_params, 'vocab_size': tokenizer.vocab_size})

    best_state, tokens_since = None, 0
    sync = torch.cuda.synchronize if device_type == 'cuda' else (lambda: None)
    t_start = time.time() - (history['elapsed'][-1] if history['elapsed'] else 0.0)
    t_last = time.time()
    for it in range(start_it, cfg.max_iters + 1):
        lr = get_lr(it, cfg)
        for g in optimizer.param_groups:
            g['lr'] = lr

        if it % cfg.eval_interval == 0 or it == cfg.max_iters:
            sync()
            interval = time.time() - t_last
            tps = tokens_since / interval if tokens_since > 0 else 0.0  # throughput of the training steps only, the evaluation below isn't counted
            losses = estimate_loss(model, train_data, val_data, cfg, device, amp_ctx)
            bpc = bits_per_char(losses['val'], len(val_data), val_chars)
            for k, v in zip(history.keys(), [it, losses['train'], losses['val'], perplexity(losses['val']), bpc, lr, tps, time.time() - t_start]):
                history[k].append(v)
            if losses['val'] < history['best_val']:
                history['best_val'] = losses['val']
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}  # keep the best weights, the last ones may already be overfit
            overfit = detector.update(losses['train'], losses['val'])
            if overfit and history['overfit_step'] is None:
                history['overfit_step'] = it
                if verbose:
                    print(f"  -> overfitting detected at step {it}: val loss stopped improving while train loss kept dropping (gap {losses['val'] - losses['train']:.3f})")
            if verbose:
                print(f"step {it} (epoch {it / iters_per_epoch:.1f}): train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, val ppl {perplexity(losses['val']):.2f}, val bpc {bpc:.3f}, lr {lr:.2e}, {tps:,.0f} tok/s")
            if cfg.use_wandb:
                wandb.log({'train_loss': losses['train'], 'val_loss': losses['val'], 'val_ppl': perplexity(losses['val']), 'val_bpc': bpc,
                           'lr': lr, 'tokens_per_sec': tps, 'gap': losses['val'] - losses['train'], 'epoch': it / iters_per_epoch}, step=it)
            if overfit and cfg.early_stop:
                break
            tokens_since, t_last = 0, time.time()

        if it == cfg.max_iters:
            break

        x, y = get_batch(train_data, cfg, device)
        with amp_ctx():  # only the forward pass runs in 16 bit, the weights and the optimizer stay in fp32
            logits, loss = fwd(x, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if cfg.grad_clip > 0:
            scaler.unscale_(optimizer)  # clipping must see the real gradients, not the scaled ones
            # if one unlucky batch produces a huge gradient, we rescale it so the update can't blow up the weights
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        tokens_since += cfg.batch_size * cfg.chunk_size

        if ckpt_every and (it + 1) % ckpt_every == 0:
            epoch = (it + 1) / iters_per_epoch
            path = ckpt_path.replace('.pt', f'-epoch{epoch:.2f}'.rstrip('0').rstrip('.') + '.pt')
            last = {'val_loss': history['val_loss'][-1], 'val_ppl': history['val_ppl'][-1], 'val_bpc': history['val_bpc'][-1], 'n_params': n_params, 'step': it + 1}  # from the latest evaluation
            weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            save_checkpoint(path, weights, cfg, tokenizer, last, optimizer, it, history)
            if verbose:
                print(f"  -> saved {path} (epoch {epoch:.2f})")

    history['train_time'] = time.time() - t_start
    if ckpt_path:
        b = min(range(len(history['val_loss'])), key=lambda i: history['val_loss'][i])
        metrics = {'val_loss': history['val_loss'][b], 'val_ppl': history['val_ppl'][b], 'val_bpc': history['val_bpc'][b], 'n_params': n_params, 'step': history['step'][b]}
        save_checkpoint(ckpt_path, best_state if best_state is not None else model.state_dict(), cfg, tokenizer, metrics)
    if cfg.use_wandb:
        sample = tokenizer.decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 300)[0].tolist())
        wandb.log({'sample': wandb.Html(f"<pre>{sample}</pre>")})
        wandb.finish()
    return model, history

In [ ]:
# the new training options, all of them are just config fields (and command line flags in the project)
# - mixed_precision: forward pass in 16 bit on the gpu, usually ~1.5-3x faster and half the activation memory (bf16 on ampere+, fp16 with loss scaling on older gpus like the kaggle T4)
# - compile_model: torch.compile, cuda only, ~1 minute of compile time at the start, then faster steps
# - ckpt_every_epochs: pass ckpt_path to train_model and a resumable checkpoint (weights + optimizer + step) is written every N epochs, resume with resume_from=that_file
# - data_path: the file / folder / url that load_text() reads
# fast_cfg = Config(**PRESETS['medium']).but(mixed_precision=True, compile_model=True, ckpt_every_epochs=10)
# model, history = train_model(fast_cfg, tokenizer, train_data, val_data, val_chars, device, ckpt_path='char-medium.pt')
# model, history = train_model(fast_cfg, tokenizer, train_data, val_data, val_chars, device, ckpt_path='char-medium.pt', resume_from='char-medium-epoch10.pt')

# one proper run with the default (medium) config
cfg = Config(**PRESETS['medium'])
tokenizer, train_data, val_data, val_chars = prepare(cfg, text)
model, history = train_model(cfg, tokenizer, train_data, val_data, val_chars, device, name='medium-char')

idx = torch.tensor([tokenizer.encode("\n")], dtype=torch.long, device=device)
print(tokenizer.decode(model.generate(idx, max_new_tokens=500)[0].tolist()))

# the loss curves, the gap between the two is the overfitting
plt.plot(history['step'], history['train_loss'], label='train'); plt.plot(history['step'], history['val_loss'], label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.show()

## 7. experiments

In [ ]:
# the experiment runner: a study is a list of (label, overrides), every run differs from the base config in ONE thing
# the summary table is the one from the ideas list, filled in by real runs

# every study is just a list of (label, config overrides), the runs differ in ONE thing so the comparison means something
STUDIES = {
    "lr_schedule": [
        ("constant", dict(lr_schedule="constant")),
        ("cosine", dict(lr_schedule="cosine")),
        ("warmup+cosine", dict(lr_schedule="warmup_cosine")),
    ],
    "optimizer": [
        ("adam", dict(optimizer="adam")),
        ("adamw", dict(optimizer="adamw")),
        ("adamw, no clip", dict(optimizer="adamw", grad_clip=0.0)),
        ("adamw, lr 1e-3", dict(optimizer="adamw", learning_rate=1e-3)),
    ],
    "tokenizer": [
        ("char", dict(tokenizer="char")),
        ("word", dict(tokenizer="word")),
        ("bpe", dict(tokenizer="bpe")),
    ],
    "architecture": [
        ("2 layers / 2 heads / 64", dict(n_layer=2, n_head=2, n_embd=64)),
        ("4 layers / 4 heads / 128", dict(n_layer=4, n_head=4, n_embd=128)),
        ("6 layers / 4 heads / 256", dict(n_layer=6, n_head=4, n_embd=256)),
    ],
    "components": [
        ("relu + learned pos (baseline)", dict(activation="relu", pos_encoding="learned", weight_tying=False)),
        ("gelu", dict(activation="gelu", pos_encoding="learned", weight_tying=False)),
        ("gelu + weight tying", dict(activation="gelu", pos_encoding="learned", weight_tying=True)),
        ("gelu + tying + rope", dict(activation="gelu", pos_encoding="rope", weight_tying=True)),
        ("post-ln", dict(pre_ln=False)),
    ],
}


def run_study(study, base_cfg, text, device, verbose=True):
    results = []
    for label, overrides in STUDIES[study]:
        cfg = base_cfg.but(**overrides)
        tokenizer, train_data, val_data, val_chars = prepare(cfg, text)
        model, history = train_model(cfg, tokenizer, train_data, val_data, val_chars, device, name=f"{study}: {label}", verbose=verbose)
        history['label'] = label
        history['sample'] = generate_text(model, tokenizer, "\n", 200, device)
        results.append(history)
    return results


def summarize(results):
    # the table from the ideas list, but with numbers that come from real runs
    header = f"{'run':32s} {'params':>9s} {'val loss':>9s} {'val ppl':>9s} {'val bpc':>8s} {'tok/s':>9s} {'time(s)':>8s}  overfit@"
    lines = [header, '-' * len(header)]
    for h in results:
        best = min(range(len(h['val_loss'])), key=lambda i: h['val_loss'][i])
        tps = [t for t in h['tokens_per_sec'] if t > 0]
        lines.append(f"{h['label']:32s} {h['n_params']:9,d} {h['val_loss'][best]:9.4f} {h['val_ppl'][best]:9.2f} {h['val_bpc'][best]:8.3f} "
                     f"{sum(tps) / max(1, len(tps)):9,.0f} {h['train_time']:8.1f}  {h['overfit_step']}")
    return '\n'.join(lines)


def plot_results(results, title="", save_to=None):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for h in results:
        axes[0].plot(h['step'], h['train_loss'], label=h['label'])
        axes[1].plot(h['step'], h['val_loss'], label=h['label'])
        axes[2].plot(h['step'], h['val_bpc'], label=h['label'])
    for ax, t in zip(axes, ['train loss', 'val loss', 'val bits per char']):
        ax.set_title(f"{title} - {t}")
        ax.set_xlabel('step')
        ax.legend(fontsize=7)
    plt.tight_layout()
    if save_to:
        fig.savefig(save_to, dpi=120)
    return fig

In [ ]:
# study 1: constant lr vs cosine decay vs warmup + cosine decay (same seed, same everything else)
# the studies below use 2000 steps so they finish in reasonable time, raise it if you have a gpu to spare
base = Config(max_iters=2000, eval_interval=200, warmup_iters=80)
res_lr = run_study('lr_schedule', base, text, device)
print(summarize(res_lr)); plot_results(res_lr, 'lr schedule'); plt.show()

In [ ]:
# study 2: adam vs adamw, with and without gradient clipping, and a higher learning rate
res_opt = run_study('optimizer', base, text, device)
print(summarize(res_opt)); plot_results(res_opt, 'optimizer'); plt.show()

In [ ]:
# study 3: tokenizers. same architecture and same number of steps, so same compute per step, we compare on val bits per char
# keep in mind: the word model has a much bigger embedding matrix (see params), and its bpc is optimistic because every <unk> hides the characters of a rare word
# also with chunk_size tokens the bpe model sees ~2x more characters of context than the char model, and the word model ~3.7x, that is part of the point of bigger tokens
res_tok = run_study('tokenizer', base, text, device)
print(summarize(res_tok)); plot_results(res_tok, 'tokenizer'); plt.show()
for h in res_tok: print('---', h['label']); print(h['sample'])

In [ ]:
# study 4: architecture sizes, this is the A/B/C table from the ideas list
res_arch = run_study('architecture', base, text, device)
print(summarize(res_arch)); plot_results(res_arch, 'architecture'); plt.show()

In [ ]:
# study 5: the components, one change at a time: gelu, weight tying, rope, post-ln
res_comp = run_study('components', base, text, device)
print(summarize(res_comp)); plot_results(res_comp, 'components'); plt.show()

## 8. attention visualization

In [ ]:
# one heatmap per head: row = the token that is looking, column = the token it is looking at, the upper triangle is black because of the causal mask
# the sentence is encoded with the model's own tokenizer, so a bpe model shows subword pieces

def token_labels(tokenizer, ids):
    return [tokenizer.decode([i]).replace('\n', '\\n').replace(' ', '·') for i in ids]  # make newlines and spaces visible on the axes


def plot_attention(model, tokenizer, sentence, layer=0, device='cpu'):
    # one heatmap per head: row = the token that is looking, column = the token it is looking at
    ids = tokenizer.encode(sentence)[-model.cfg.chunk_size:]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    maps = model.attention_maps(idx)  # (n_layer, n_head, T, T)
    n_head = maps.size(1)
    labels = token_labels(tokenizer, ids)
    fig, axes = plt.subplots(1, n_head, figsize=(4.2 * n_head, 4.4), squeeze=False)
    for h in range(n_head):
        ax = axes[0][h]
        ax.imshow(maps[layer, h].numpy(), cmap='viridis')
        ax.set_title(f"layer {layer}, head {h}")
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90, fontsize=6)
        ax.set_yticklabels(labels, fontsize=6)
    plt.tight_layout()
    return fig

model.eval()
for layer in range(cfg.n_layer):
    plot_attention(model, tokenizer, "To be, or not to be, that is the question.", layer, device); plt.show()

# for an interactive version in the notebook:
# import ipywidgets as w
# w.interact(lambda layer: (plot_attention(model, tokenizer, "To be, or not to be, that is the question.", layer, device), plt.show()), layer=(0, cfg.n_layer - 1))

## 9. kv cache

In [ ]:
# generation used to recompute attention for the whole context at every new token, with the cache we compute keys and values of each token once
# the cache is exact (we verified that above), so the only thing that changes is the speed
print('cache matches full forward pass:', verify_cache(model, device))
benchmark_kv_cache(model, device=device)

# beyond chunk_size tokens the window is full and the cache is rebuilt from the last half of the context, let us see that this still works
print(tokenizer.decode(model.generate(idx, max_new_tokens=cfg.chunk_size * 3)[0].tolist())[:300])

## 10. sampling controls (temperature and top-k)

In [ ]:
# until now generate() always sampled from the raw softmax, the two standard knobs live inside generate() in gpt.py (see the generate cell above)
# - temperature divides the logits: <1 makes the model more confident and repetitive, >1 makes it more adventurous and sloppier, 0 means greedy (always the most likely token)
# - top-k keeps only the k most likely tokens and throws away the long tail of garbage, this is what stops one unlucky sample from derailing a whole sentence
# both only touch the last logits, so they work with the kv cache without any change
prompt = torch.tensor([tokenizer.encode("ROMEO:")], dtype=torch.long, device=device)
for t, k in [(0, None), (0.5, None), (1.0, None), (1.5, None), (1.0, 10), (1.0, 1)]:
    torch.manual_seed(1337)
    print(f"--- temperature {t}, top_k {k}")
    print(tokenizer.decode(model.generate(prompt, 150, temperature=t, top_k=k)[0].tolist()))

# sanity check: top_k=1 is greedy decoding, so it has to be identical to temperature=0
print(torch.equal(model.generate(prompt, 100, temperature=0), model.generate(prompt, 100, top_k=1)))

## 11. experiment tracking

In [ ]:
# weights & biases: set use_wandb=True and everything the loop already computes (config, losses, perplexity, bits per char, lr, throughput, a generated sample)
# is logged, and the gpu utilization is tracked automatically by wandb itself
# !pip install wandb
# import wandb; wandb.login()
# cfg = Config(**PRESETS['medium']).but(use_wandb=True)
# tokenizer, train_data, val_data, val_chars = prepare(cfg, text)
# model, history = train_model(cfg, tokenizer, train_data, val_data, val_chars, device, name='medium-char')

## 11. next: leave the notebook
this notebook stays as the explanation. the same code (literally the same, the cells above are generated from the files) lives in a project:

```
configs/   model/   data/   training/   inference/   experiments/   app/   checkpoints/   notebooks/
```
see the README in the project for the commands.